### About Dataset
This dataset comprises customer reviews for Amazon, an online retail giant, featuring insights into customer experiences, including ratings, review titles, texts, and metadata. It is valuable for analyzing customer satisfaction, sentiment, and trends.

## Column Descriptions:

Reviewer Name: Identifies the reviewer.
rofile Link: Links to the reviewer's profile for additional insights.
Country: Indicates the reviewer's location.
Review Count: Number of reviews by the same user, showing engagement level.
Review Date: When the review was posted, useful for time analysis.
Rating: Numerical satisfaction measure.
Review Title: Summarizes the review sentiment.
Review Text: Detailed customer feedback.
Date of Experience: When the service/product was experienced.

## Prospective applications:

Sentiment Analysis: Analyze review texts and titles to assess overall customer sentiment toward products, enabling the identification of strengths and weaknesses.
Customer Satisfaction Tracking: Track and visualize rating trends over time to understand fluctuations in customer satisfaction.
Product Improvement: Identify common themes in reviews to highlight areas for product enhancement or development.
Market Segmentation: Use country and demographic information to customize marketing strategies and gain insights into regional preferences.
Competitor Analysis: Evaluate customer feedback on Amazon products in comparison to competitors to determine market positioning.
Recommendation Systems: Leverage review data to enhance recommendation algorithms, improving personalized shopping experiences.
Trend Analysis: Investigate temporal patterns in reviews to link sentiment changes with marketing efforts or product launches.

This extensive dataset serves as a valuable asset for various analyses focused on enhancing customer engagement and refining business strategies.

In [1]:
# libraries
import numpy as np 
import pandas as pd

In [20]:
# importing the dataset
amazon_reviews = pd.read_csv('Amazon_Reviews.csv', encoding='utf-8', engine='python')

# checking the dataset
print(amazon_reviews.head())

      Reviewer Name                     Profile Link Country Review Count  \
0        Eugene ath  /users/66e8185ff1598352d6b3701a      US     1 review   
1  Daniel ohalloran  /users/5d75e460200c1f6a6373648c      GB    9 reviews   
2          p fisher  /users/546cfcf1000064000197b88f      GB   90 reviews   
3         Greg Dunn  /users/62c35cdbacc0ea0012ccaffa      AU    5 reviews   
4     Sheila Hannah  /users/5ddbe429478d88251550610e      GB    8 reviews   

                Review Date                  Rating  \
0  2024-09-16T13:44:26.000Z  Rated 1 out of 5 stars   
1  2024-09-16T18:26:46.000Z  Rated 1 out of 5 stars   
2  2024-09-16T21:47:39.000Z  Rated 1 out of 5 stars   
3  2024-09-17T07:15:49.000Z  Rated 1 out of 5 stars   
4  2024-09-16T18:37:17.000Z  Rated 1 out of 5 stars   

                                      Review Title  \
0       A Store That Doesn't Want to Sell Anything   
1           Had multiple orders one turned up and…   
2                      I informed these repr

## Task 1: Preprocessing

In [21]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADNANLAPTOP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADNANLAPTOP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADNANLAPTOP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [22]:
# Preprocessing function
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Tokenization
    tokens = nltk.word_tokenize(text)
    
    # 3. Remove punctuation
    tokens = [word for word in tokens if word.isalnum()]
    
    # 4. Optional: Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # 5. Optional: Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

# Apply preprocessing to review text
amazon_reviews['processed_text'] = amazon_reviews['Review Text'].apply(preprocess_text)

print("Sample processed text:")
print(amazon_reviews['processed_text'].head())

Sample processed text:
0    registered website tried order laptop entered ...
1    multiple order one turned driver phone door nu...
2    informed reprobate would going visit sick rela...
3    bought amazon problem happy service price amaz...
4    could give lower rate would cancelled amazon p...
Name: processed_text, dtype: str


## Task 2: Vocabulary Creation

In [12]:
# Build vocabulary using CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(amazon_reviews['processed_text'])

# Vocabulary
vocabulary = vectorizer.get_feature_names_out()
vocab_size = len(vocabulary)

print(f"Vocabulary size: {vocab_size}")

# Top frequent words
word_freq = X_bow.sum(axis=0).A1
word_freq_dict = dict(zip(vocabulary, word_freq))
sorted_words = sorted(word_freq_dict.items(), key=lambda x: x[1], reverse=True)

print("Top 10 frequent words:")
for word, freq in sorted_words[:10]:
    print(f"{word}: {freq}")

Vocabulary size: 24902
Top 10 frequent words:
amazon: 30823
customer: 12432
service: 11565
item: 9838
delivery: 8538
time: 8328
order: 7909
day: 7819
get: 7637
prime: 5768


## Task 3: Feature Engineering

In [13]:
# 1. One Hot Encoding (binary vector)
ohe_vectorizer = CountVectorizer(binary=True)
X_ohe = ohe_vectorizer.fit_transform(amazon_reviews['processed_text'])

# 2. Bag of Words using CountVectorizer
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(amazon_reviews['processed_text'])

# 3. TF-IDF using TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(amazon_reviews['processed_text'])

print("Feature matrices created:")
print(f"OHE shape: {X_ohe.shape}")
print(f"BoW shape: {X_bow.shape}")
print(f"TF-IDF shape: {X_tfidf.shape}")

Feature matrices created:
OHE shape: (21214, 24902)
BoW shape: (21214, 24902)
TF-IDF shape: (21214, 24902)


## Task 4: Comparison Analysis

In [14]:
# Comparison table
import pandas as pd

# Sample document
sample_doc = amazon_reviews['processed_text'].iloc[0]
print(f"Sample document: {sample_doc}")

# Get vectors for sample
ohe_sample = X_ohe[0].toarray().flatten()
bow_sample = X_bow[0].toarray().flatten()
tfidf_sample = X_tfidf[0].toarray().flatten()

# Get feature names
ohe_features = ohe_vectorizer.get_feature_names_out()
bow_features = bow_vectorizer.get_feature_names_out()
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Create DataFrame for non-zero values
comparison_data = []
for i, word in enumerate(ohe_features):
    if ohe_sample[i] > 0 or bow_sample[i] > 0 or tfidf_sample[i] > 0:
        comparison_data.append({
            'Word': word,
            'OHE': ohe_sample[i],
            'BoW': bow_sample[i],
            'TF-IDF': tfidf_sample[i]
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.head(10))

# Explanation
print("\nExplanation:")
print("In TF-IDF, words that appear frequently across documents (like common words) get lower weights because they are less informative.")
print("Rare words that appear in few documents get higher weights.")
print("Most important words in TF-IDF are those with high TF-IDF scores, indicating they are unique to the document and not common.")

Sample document: registered website tried order laptop entered detail instead charging sending product froze account demanding various verification document sent said would review within 24 hour reality week one help give truthful estimate resolved tell never seen horrible marketplace life hope came ca buy food store receiving review request take forever process
        Word  OHE  BoW    TF-IDF
0         24    1    1  0.149901
1    account    1    1  0.084023
2        buy    1    1  0.094666
3         ca    1    1  0.104892
4       came    1    1  0.121304
5   charging    1    1  0.155554
6  demanding    1    1  0.223039
7     detail    1    1  0.139210
8   document    1    1  0.176215
9    entered    1    1  0.191678

Explanation:
In TF-IDF, words that appear frequently across documents (like common words) get lower weights because they are less informative.
Rare words that appear in few documents get higher weights.
Most important words in TF-IDF are those with high TF-IDF scores, in

## Task 5: Sparse Matrix Analysis

In [15]:
# Sparse matrix analysis
def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.nnz
    sparsity = (1 - non_zero / total_elements) * 100
    return sparsity

print(f"OHE Matrix shape: {X_ohe.shape}, Sparsity: {calculate_sparsity(X_ohe):.2f}%")
print(f"BoW Matrix shape: {X_bow.shape}, Sparsity: {calculate_sparsity(X_bow):.2f}%")
print(f"TF-IDF Matrix shape: {X_tfidf.shape}, Sparsity: {calculate_sparsity(X_tfidf):.2f}%")

print("\nExplanation:")
print("Sparse matrices have many zeros, wasting memory and computation time in dense operations.")
print("For large-scale systems, they are inefficient because most operations don't benefit from the zeros.")
print("Specialized sparse matrix libraries are needed for efficiency.")

OHE Matrix shape: (21214, 24902), Sparsity: 99.87%
BoW Matrix shape: (21214, 24902), Sparsity: 99.87%
TF-IDF Matrix shape: (21214, 24902), Sparsity: 99.87%

Explanation:
Sparse matrices have many zeros, wasting memory and computation time in dense operations.
For large-scale systems, they are inefficient because most operations don't benefit from the zeros.
Specialized sparse matrix libraries are needed for efficiency.


## Task 6: Real-world Questions

In [16]:
# Answers to real-world questions
print("1. Why Bag of Words fails in understanding semantic meaning:")
print("BoW treats words as independent units, ignoring context, synonyms, and polysemy.")
print("Example: 'bank' in 'river bank' vs 'bank account' have different meanings but same representation.")
print("It can't capture relationships like negation or word order.")

print("\n2. When to use Bag of Words and TF-IDF in industry:")
print("BoW: Simple tasks like spam detection, basic classification where frequency matters.")
print("TF-IDF: Document retrieval, search engines, content-based recommendation, where term importance varies.")

print("\n3. Limitations of TF-IDF in real applications:")
print("Doesn't capture semantics or word relationships.")
print("Ignores word order and context.")
print("Struggles with out-of-vocabulary words.")
print("Not suitable for languages with complex morphology.")
print("Requires large corpora for good IDF estimates.")

1. Why Bag of Words fails in understanding semantic meaning:
BoW treats words as independent units, ignoring context, synonyms, and polysemy.
Example: 'bank' in 'river bank' vs 'bank account' have different meanings but same representation.
It can't capture relationships like negation or word order.

2. When to use Bag of Words and TF-IDF in industry:
BoW: Simple tasks like spam detection, basic classification where frequency matters.
TF-IDF: Document retrieval, search engines, content-based recommendation, where term importance varies.

3. Limitations of TF-IDF in real applications:
Doesn't capture semantics or word relationships.
Ignores word order and context.
Struggles with out-of-vocabulary words.
Not suitable for languages with complex morphology.
Requires large corpora for good IDF estimates.


## Task 7: Mini Use Case - Sentiment Classification

In [18]:
# Prepare labels
amazon_reviews['sentiment'] = amazon_reviews['Rating'].apply(lambda x: 1 if isinstance(x, str) and ('5' in x or '4' in x) else 0)  # 1: positive, 0: negative

# Split data
X_train, X_test, y_train, y_test = train_test_split(amazon_reviews['processed_text'], amazon_reviews['sentiment'], test_size=0.2, random_state=42)

# Vectorize
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Train and evaluate Logistic Regression
lr_bow = LogisticRegression()
lr_bow.fit(X_train_bow, y_train)
y_pred_lr_bow = lr_bow.predict(X_test_bow)

lr_tfidf = LogisticRegression()
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

# Train and evaluate Naive Bayes
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
y_pred_nb_bow = nb_bow.predict(X_test_bow)

nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

# Print results
print("Logistic Regression with BoW:")
print(classification_report(y_test, y_pred_lr_bow))

print("Logistic Regression with TF-IDF:")
print(classification_report(y_test, y_pred_lr_tfidf))

print("Naive Bayes with BoW:")
print(classification_report(y_test, y_pred_nb_bow))

print("Naive Bayes with TF-IDF:")
print(classification_report(y_test, y_pred_nb_tfidf))

Logistic Regression with BoW:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        28
           1       0.99      1.00      1.00      4215

    accuracy                           0.99      4243
   macro avg       0.50      0.50      0.50      4243
weighted avg       0.99      0.99      0.99      4243

Logistic Regression with TF-IDF:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        28
           1       0.99      1.00      1.00      4215

    accuracy                           0.99      4243
   macro avg       0.50      0.50      0.50      4243
weighted avg       0.99      0.99      0.99      4243

Naive Bayes with BoW:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        28
           1       0.99      1.00      1.00      4215

    accuracy                           0.99      4243
   macro avg       0.50      0.50      0.50